In [ ]:
import pandas as pd
import numpy as np

# Load data
phys = pd.read_csv("physiological_cycles.csv")
slp  = pd.read_csv("sleeps.csv")
wkt  = pd.read_csv("workouts.csv")

# Convert timestamps to datetimes and extract a daily key
# For physiological and sleep data, use "Cycle start time" as the reference
phys["Cycle start time"] = pd.to_datetime(phys["Cycle start time"], errors="coerce")
slp["Cycle start time"]  = pd.to_datetime(slp["Cycle start time"],  errors="coerce")

phys["day"] = phys["Cycle start time"].dt.date
slp["day"]  = slp["Cycle start time"].dt.date

# For workouts, use "Workout start time" as the reference
wkt["Workout start time"] = pd.to_datetime(wkt["Workout start time"], errors="coerce")
wkt["day"] = wkt["Workout start time"].dt.date

# Create a sorted list
# Which is going to be the index for all parameters
unique_days = sorted(set(phys["day"]) | set(slp["day"]) | set(wkt["day"]))

# steps: daily step count.
# empty placeholder
steps = np.zeros(len(unique_days))

# tib: "time in bed" per day in HOURS.
# "In bed duration (min)" summed per day.
tib = (
    slp.groupby("day")["In bed duration (min)"]
       .sum()
       .reindex(unique_days, fill_value=0)
       .to_numpy() / 60.0
)

# ac: "active calories burned" per day in CALORIES.
# "Energy burned (kcal)" summed per day.
ac = (
    phys.groupby("day")["Energy burned (cal)"]
        .sum()
        .reindex(unique_days, fill_value=0)
        .to_numpy() / 1000.0
)

# exm: "exercise minutes" per day in MINUTES.
# "Duration (min)" for exercise summed per day.
exm = (
    wkt.groupby("day")["Duration (min)"]
       .sum()
       .reindex(unique_days, fill_value=0)
       .to_numpy()
)

# nsed: "non-sedentary hours" per day in HOURS.
# total hours in a day minus time spent in be
nsed = 24 - tib

# dlh: "daylight hours" per day in HOURS.
# empty placeholder
dlh = np.zeros(len(unique_days))

# dist: "distance walked" per day in METERS.
# empty placeholder
dist = np.zeros(len(unique_days))

# flt: "flights of stairs climbed" per day (COUNT).
# empty placeholder
flt = np.zeros(len(unique_days))

# rhr: "average resting heart rate" per day in BPM.
# "Resting heart rate (bpm)".
rhr = (
    phys.groupby("day")["Resting heart rate (bpm)"]
        .mean()
        .reindex(unique_days, fill_value=0)
        .to_numpy()
)

# Stack everything into X_raw in order:
X_raw = np.vstack([steps, tib, ac, exm, nsed, dlh, dist, flt, rhr]).T

print("X_raw shape:", X_raw.shape)
print("First few rows of X_raw:\n", X_raw[:5])
